In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
# data visualixxaton
import matplotlib.pyplot as plt

import cv2
AUTOTUNE = tf.data.AUTOTUNE


In [74]:
BATCH_SIZE = 32
IMAGE_SIZE = 256
CHANNELS=3
EPOCHS=50

In [ ]:
dataDir = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    seed=123,
    shuffle=True,
    image_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

print(dataDir)

# column name
column_name = dataDir.class_names

print(np.array(column_name))


In [ ]:
len(dataDir)
# 556 * 10
# print(1032 * 20)

In [ ]:
# train dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

In [ ]:
x = []
y=  []

for image_batch, label_batch in train_ds:
    # for clear img convert to int
    plt.imshow(image_batch[0].numpy().astype("uint8"))
    plt.axis("off")
    plt.title(column_name[label_batch[0]])
    plt.show()
    x.append(image_batch[0].numpy().astype("uint8"))
    y.append(column_name[label_batch[0]])

In [ ]:
x[0]

plt.imshow(x[0])
plt.axis("off")
plt.title(column_name[label_batch[0]])
plt.show()


In [ ]:
class_names = dataDir.class_names
class_names

In [ ]:
print(len(x))
print(len(y))

In [ ]:
len(dataDir)

In [ ]:
train_size = 0.8
len(dataDir)*train_size

In [ ]:
train_ds_2 = dataDir.take(54)
len(train_ds)

In [ ]:
test_ds_2 = dataDir.skip(54)
len(test_ds_2)

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
val_size=0.1
len(dataDir)*val_size

In [ ]:
test_ds_2 = test_ds_2.skip(6)
len(test_ds_2)

In [ ]:
# pre proceesing
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.2),
])

resize_rescale = keras.Sequential([
    keras.layers.Resizing(IMAGE_SIZE,IMAGE_SIZE),
    keras.layers.Rescaling(1./255),
])

In [ ]:
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
input_shape = (BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
n_classes = 3

model = keras.models.Sequential([
    resize_rescale,
    # data_augmentation,
    keras.layers.Conv2D(32, kernel_size = (3,3), activation='relu', input_shape=input_shape),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64,  kernel_size = (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64,  kernel_size = (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(len(column_name), activation='softmax'),
])

model.build(input_shape=input_shape)

In [ ]:
# model.build(input_shape=(BATCH_SIZE,IMAGE_SIZE,IMAGE_SIZE,CHANNELS))

model.compile(
    optimizer='adam',
    # loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

In [ ]:
for images, labels in train_ds.take(1):
    print(images.shape, labels.shape)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    epochs=50,
    verbose=1,
    validation_data=val_ds
)


In [ ]:
model.save(r"E:\python\tenserflow\Exercises\potato_healthy_plant_detection\model_healty_plant\plant_status_detector_v3.keras")

In [2]:
model_load = keras.models.load_model(r"E:\python\tenserflow\Exercises\potato_healthy_plant_detection\model_healty_plant\plant_status_detector_v2.keras")

In [77]:
model_load.evaluate(val_ds)

129/129 ━━━━━━━━━━━━━━━━━━━━ 16s 120ms/step - accuracy: 0.9416 - loss: 0.2401


[0.24008186161518097, 0.9416040778160095]

In [78]:
# train dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    validation_split=0.3,   # 30% reserved for val+test
    subset="training",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
)


val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    validation_split=0.3,   # same 30% reserved
    subset="validation",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
)

val_batches = tf.data.experimental.cardinality(val_ds)
test_size = val_batches // 3   # roughly 10% of total dataset

test_ds = val_ds.take(test_size)       # first 10% → test
val_ds = val_ds.skip(test_size)        # remaining 20% → validation



Found 20638 files belonging to 15 classes.
Using 14447 files for training.
Found 20638 files belonging to 15 classes.
Using 6191 files for validation.


In [79]:
model_load.evaluate(test_ds)

64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 113ms/step - accuracy: 0.9961 - loss: 0.0152


[0.015156646259129047, 0.99609375]

In [ ]:
img_input = cv2.imread(r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage\Tomato_Late_blight\b0a1ada0-f496-46a3-827e-2d721dabee59___GHLB2 Leaf 9086.JPG")

img_resized_2 = cv2.resize(img_input, (IMAGE_SIZE, IMAGE_SIZE))   # Resize to model input size
img_resized_2 = img_resized_2 / 255.0 

# Add batch dimension (model expects 4D input)
img_input_f = np.expand_dims(img_resized_2, axis=0)    

print(np.array(column_name))

# model_load.predict(img_input_f)
prediction = model_load.predict(img_input_f)

val = np.argmax(prediction)
val

print(column_name[val])

In [ ]:
def model_predict(image_path):
    img_input_2 = cv2.imread(str(image_path))

    input_img_21 = tf.keras.preprocessing.image.img_to_array(img_input_2)
    input_img_21_array = tf.expand_dims(input_img_21, 0)

    prediction = model_load.predict(input_img_21_array)
    prediction

    pred_val = column_name[np.argmax(prediction)]

    confidence = round(100 * (np.max(prediction[0])))

    plt.imshow(cv2.cvtColor(img_input_2, cv2.COLOR_BGR2RGB))  # show original image
    plt.axis("off")
    plt.title(f"{pred_val} ({confidence}%)")
    plt.show()

    print(pred_val)
    print(confidence)

    return pred_val, confidence

In [ ]:
model_predict(r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage\Tomato_healthy\a261cf1c-0bf5-4dec-b796-5dfe2ab0ee1b___GH_HL Leaf 322.JPG")


In [ ]:
model_predict(r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage\Tomato_Septoria_leaf_spot\ca8982f7-f569-4f25-a4c5-7a4090d8418f___Matt.S_CG 7461.JPG")

In [ ]:
model_predict(r"C:\Users\YASHASH RAVI\Downloads\WhatsApp Image 2025-11-21 at 09.55.40 (1).jpeg")

In [82]:
model_load.evaluate(dataDir)

645/645 ━━━━━━━━━━━━━━━━━━━━ 69s 107ms/step - accuracy: 0.9855 - loss: 0.0583


[0.05826807767152786, 0.9854636788368225]

In [80]:
model_load.evaluate(test_ds)


64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 112ms/step - accuracy: 0.9961 - loss: 0.0153


[0.015303421765565872, 0.99609375]

In [81]:
model_load.evaluate(val_ds)

130/130 ━━━━━━━━━━━━━━━━━━━━ 15s 109ms/step - accuracy: 0.9435 - loss: 0.2356


[0.23555538058280945, 0.9435191750526428]